In [6]:
import os
import struct
import math
import numpy as np
import cv2  # hanya dipakai untuk cv2.imread (membaca file gambar) -> ini TIDAK dilarang
from tqdm import tqdm

# 1. Tentukan folder asal dan folder tujuan penyimpanan
PATH_INPUT = "D:/semester 4/PCD/praktikum/projek/test/Assets/"
PATH_OUTPUT = "D:/semester 4/PCD/praktikum/projek/test/Assets_Prepro3/"
KATEGORI = ["Normal", "kidneyStone"]


# ======================================================================
# FUNGSI 1: RESIZE MANUAL  (pengganti cv2.resize)
# Memakai interpolasi bilinear -> hasilnya sama persis dengan cara kerja
# default cv2.resize (INTER_LINEAR), hanya saja dihitung sendiri pakai numpy.
# ======================================================================
def resize_manual(img, ukuran_baru):
    tinggi_lama, lebar_lama = img.shape
    lebar_baru, tinggi_baru = ukuran_baru   # urutan (lebar, tinggi) sama seperti cv2.resize
    img_f = img.astype(np.float64)

    skala_x = lebar_lama / lebar_baru
    skala_y = tinggi_lama / tinggi_baru

    # koordinat tiap piksel baru, dipetakan balik ke koordinat gambar lama
    y_asal = np.clip((np.arange(tinggi_baru) + 0.5) * skala_y - 0.5, 0, tinggi_lama - 1)
    x_asal = np.clip((np.arange(lebar_baru) + 0.5) * skala_x - 0.5, 0, lebar_lama - 1)

    y0 = np.floor(y_asal).astype(int)
    y1 = np.minimum(y0 + 1, tinggi_lama - 1)
    wy = (y_asal - y0).reshape(-1, 1)          # bobot arah vertikal

    x0 = np.floor(x_asal).astype(int)
    x1 = np.minimum(x0 + 1, lebar_lama - 1)
    wx = (x_asal - x0).reshape(1, -1)          # bobot arah horizontal

    # ambil 4 piksel tetangga (atas-kiri, atas-kanan, bawah-kiri, bawah-kanan)
    p00 = img_f[np.ix_(y0, x0)]
    p01 = img_f[np.ix_(y0, x1)]
    p10 = img_f[np.ix_(y1, x0)]
    p11 = img_f[np.ix_(y1, x1)]

    # rumus interpolasi bilinear
    atas = p00 * (1 - wx) + p01 * wx
    bawah = p10 * (1 - wx) + p11 * wx
    hasil = atas * (1 - wy) + bawah * wy

    return np.clip(hasil, 0, 255).astype(np.uint8)


# ======================================================================
# FUNGSI 2: GAUSSIAN BLUR MANUAL  (pengganti cv2.GaussianBlur)
# ======================================================================
def buat_kernel_gaussian(ukuran=5, sigma=0):
    # rumus sigma default yang dipakai OpenCV ketika sigma=0
    if sigma is None or sigma <= 0:
        sigma = 0.3 * ((ukuran - 1) * 0.5 - 1) + 0.8

    pusat = ukuran // 2
    kernel = np.zeros((ukuran, ukuran), dtype=np.float64)
    for i in range(ukuran):
        for j in range(ukuran):
            dx, dy = i - pusat, j - pusat
            kernel[i, j] = math.exp(-(dx ** 2 + dy ** 2) / (2 * sigma ** 2))

    return kernel / kernel.sum()   # normalisasi agar total bobot = 1


def gaussian_blur_manual(img, ukuran_kernel=(5, 5), sigma=0):
    kx, ky = ukuran_kernel
    kernel = buat_kernel_gaussian(kx, sigma)
    pad = kx // 2

    # padding tepi dengan refleksi (mirip border default OpenCV)
    img_pad = np.pad(img.astype(np.float64), pad, mode='reflect')
    tinggi, lebar = img.shape
    hasil = np.zeros((tinggi, lebar), dtype=np.float64)

    # konvolusi manual: akumulasi (bobot kernel x potongan gambar bergeser)
    for i in range(kx):
        for j in range(ky):
            hasil += kernel[i, j] * img_pad[i:i + tinggi, j:j + lebar]

    return np.clip(hasil, 0, 255).astype(np.uint8)


# ======================================================================
# FUNGSI 3: CLAHE MANUAL  (pengganti cv2.createCLAHE + clahe.apply)
# Langkah: bagi gambar jadi tile -> hitung & clip histogram tiap tile ->
# equalisasi tiap tile -> interpolasi bilinear antar tile per piksel.
# ======================================================================
def clahe_manual(img, clip_limit=2.0, grid_size=(8, 8)):
    tinggi, lebar = img.shape
    gx, gy = grid_size
    tinggi_tile = tinggi // gy
    lebar_tile = lebar // gx

    # --- hitung fungsi pemetaan (hasil equalisasi) untuk setiap tile ---
    peta_tile = np.zeros((gy, gx, 256), dtype=np.float64)
    for ty in range(gy):
        y0 = ty * tinggi_tile
        y1 = tinggi if ty == gy - 1 else y0 + tinggi_tile
        for tx in range(gx):
            x0 = tx * lebar_tile
            x1 = lebar if tx == gx - 1 else x0 + lebar_tile

            tile = img[y0:y1, x0:x1]
            jumlah_piksel = tile.size

            # histogram manual (256 bin, nilai 0-255)
            hist = np.bincount(tile.ravel(), minlength=256).astype(np.float64)

            # clip limit -> potong bin yang terlalu tinggi, sebar kelebihannya rata-rata
            batas = max(1.0, clip_limit * jumlah_piksel / 256.0)
            kelebihan = np.sum(np.maximum(hist - batas, 0))
            hist = np.minimum(hist, batas)
            hist += kelebihan / 256.0

            # CDF -> fungsi pemetaan (transformasi equalisasi) tile ini
            cdf = np.cumsum(hist)
            peta_tile[ty, tx] = cdf / cdf[-1] * 255.0

    # --- interpolasi bilinear antar 4 tile terdekat untuk setiap piksel ---
    py = np.clip(np.arange(tinggi) / tinggi_tile - 0.5, 0, gy - 1)
    px = np.clip(np.arange(lebar) / lebar_tile - 0.5, 0, gx - 1)

    ty0 = np.floor(py).astype(int)
    ty1 = np.minimum(ty0 + 1, gy - 1)
    wy = (py - ty0).reshape(-1, 1)

    tx0 = np.floor(px).astype(int)
    tx1 = np.minimum(tx0 + 1, gx - 1)
    wx = (px - tx0).reshape(1, -1)

    Ty0, Tx0 = np.meshgrid(ty0, tx0, indexing='ij')
    Ty1, Tx1 = np.meshgrid(ty1, tx1, indexing='ij')

    v00 = peta_tile[Ty0, Tx0, img]
    v01 = peta_tile[Ty0, Tx1, img]
    v10 = peta_tile[Ty1, Tx0, img]
    v11 = peta_tile[Ty1, Tx1, img]

    atas = v00 * (1 - wx) + v01 * wx
    bawah = v10 * (1 - wx) + v11 * wx
    hasil = atas * (1 - wy) + bawah * wy

    return np.clip(hasil, 0, 255).astype(np.uint8)


# ======================================================================
# FUNGSI 4: IMWRITE MANUAL  (pengganti cv2.imwrite)
# Menulis langsung ke format BMP 8-bit grayscale (header dibuat manual
# dengan struct, tanpa memanggil encoder gambar apa pun).
# ======================================================================
def imwrite_manual(path, img):
    tinggi, lebar = img.shape
    img = img.astype(np.uint8)

    padding = (4 - (lebar % 4)) % 4
    ukuran_data = (lebar + padding) * tinggi
    offset_data = 14 + 40 + (256 * 4)
    ukuran_file = offset_data + ukuran_data

    # Palet: bangun array sekaligus, 1 write (bukan 256 write)
    idx = np.arange(256, dtype=np.uint8)
    palet = np.zeros((256, 4), dtype=np.uint8)
    palet[:, 0] = idx
    palet[:, 1] = idx
    palet[:, 2] = idx

    # Piksel: flip vertikal + padding sekaligus, 1 write (bukan 256 write)
    if padding > 0:
        data_pad = np.zeros((tinggi, lebar + padding), dtype=np.uint8)
        data_pad[:, :lebar] = img
        data_piksel = data_pad[::-1].tobytes()
    else:
        data_piksel = img[::-1].tobytes()

    with open(path, 'wb') as f:
        f.write(b'BM')
        f.write(struct.pack('<I', ukuran_file))
        f.write(struct.pack('<H', 0))
        f.write(struct.pack('<H', 0))
        f.write(struct.pack('<I', offset_data))
        f.write(struct.pack('<I', 40))
        f.write(struct.pack('<i', lebar))
        f.write(struct.pack('<i', tinggi))
        f.write(struct.pack('<H', 1))
        f.write(struct.pack('<H', 8))
        f.write(struct.pack('<I', 0))
        f.write(struct.pack('<I', ukuran_data))
        f.write(struct.pack('<i', 0))
        f.write(struct.pack('<i', 0))
        f.write(struct.pack('<I', 256))
        f.write(struct.pack('<I', 256))
        f.write(palet.tobytes())
        f.write(data_piksel)

    return True

# ======================================================================
# PROSES UTAMA
# ======================================================================
print("Memulai proses preprocessing gambar (versi manual)...")

for label in KATEGORI:
    folder_input = os.path.join(PATH_INPUT, label)
    folder_output = os.path.join(PATH_OUTPUT, label)

    if not os.path.exists(folder_input):
        print(f"⚠️ Melewati folder (tidak ditemukan): {folder_input}")
        continue

    os.makedirs(folder_output, exist_ok=True)

    jumlah_sukses = 0
    for nama_file in tqdm(os.listdir(folder_input), desc=label):
        if nama_file.lower().endswith(('.jpg', '.jpeg', '.png')):
            jalur_masuk = os.path.join(folder_input, nama_file)

            # ekstensi keluaran diganti .bmp karena imwrite_manual menulis
            # format BMP mentah, bukan encoder jpg/png
            nama_dasar = os.path.splitext(nama_file)[0]
            jalur_keluar = os.path.join(folder_output, nama_dasar + ".bmp")

            # --- TAHAP 1: GRAYSCALE (cv2.imread tetap dipakai, tidak dilarang) ---
            img = cv2.imread(jalur_masuk, cv2.IMREAD_GRAYSCALE)
            if img is None:
                continue

            # --- TAHAP 2: RESIZE (manual) ---
            img_resized = resize_manual(img, (256, 256))

            # --- TAHAP 3: GAUSSIAN BLUR (manual) ---
            img_blurred = gaussian_blur_manual(img_resized, (5, 5), sigma=0)

            # --- TAHAP 4: CLAHE (manual) ---
            img_clahe = clahe_manual(img_blurred, clip_limit=2.0, grid_size=(8, 8))

            # --- SIMPAN (manual, format BMP) ---
            imwrite_manual(jalur_keluar, img_clahe)
            jumlah_sukses += 1

    print(f"✅ Selesai memproses {jumlah_sukses} gambar di folder: {label}")

print(f"\n🎉 Seluruh proses selesai! Gambar hasil preprocessing ada di folder:\n{PATH_OUTPUT}")

Memulai proses preprocessing gambar (versi manual)...


Normal: 100%|██████████| 100/100 [00:01<00:00, 52.75it/s]


✅ Selesai memproses 100 gambar di folder: Normal


kidneyStone: 100%|██████████| 100/100 [00:01<00:00, 54.54it/s]

✅ Selesai memproses 100 gambar di folder: kidneyStone

🎉 Seluruh proses selesai! Gambar hasil preprocessing ada di folder:
D:/semester 4/PCD/praktikum/projek/test/Assets_Prepro3/
